In [14]:
import torch
import torch.nn as nn
import torch.optim as optim

In [15]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [16]:
torch.cuda.is_available()

True

In [17]:
torch.cuda.get_device_name(0)

'NVIDIA GeForce GTX 1050 Ti'

In [18]:
# 1. Device configuration
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")

# 2. Load MNIST dataset
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64
)

Using device: cuda


In [19]:
# 3. Define neural network
class NeuralNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.network(x)


model = NeuralNetwork().to(device)

In [20]:
# 4. Define loss function and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [21]:
# 5. Training loop
epochs = 5

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        predictions = model(images)

        loss = criterion(predictions, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {average_loss:.4f}"
    )


Epoch [1/5] Loss: 0.3440
Epoch [2/5] Loss: 0.1452
Epoch [3/5] Loss: 0.1007
Epoch [4/5] Loss: 0.0762
Epoch [5/5] Loss: 0.0610


In [22]:
# 6. Model evaluation
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        predictions = model(images)

        _, predicted = torch.max(predictions, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"\nTest Accuracy: {accuracy:.2f}%")


Test Accuracy: 97.29%


In [23]:
# 7. Save trained model
torch.save(model.state_dict(), "digits/mnist_model.pth")

print("Model saved successfully!")

Model saved successfully!
